In [ ]:
#%% HEADER

# Modules
import numpy as np
import pandas as pd
import itertools
import destruction_models as models
import tensorflow as tf

from destruction_utilities import *
from destruction_statistics import *
from numpy import random
import matplotlib.pyplot as plt
from keras import callbacks, preprocessing
from sklearn.metrics import precision_recall_curve, roc_auc_score
from os import path

In [ ]:
CITY = 'aleppo'

In [ ]:
def get_images_and_labels(CITY):
    # Reads images (subset)
    images = search_data(pattern(city=CITY, type='image'))
    images = np.array(list(map(read_raster, images)))
    images = tile_sequences(images, tile_size=(128, 128)) 
    
    # Reads labels (subset)
    labels = search_data(pattern(city=CITY, type='label'))
    labels = np.array(list(map(lambda x: read_raster(x, dtype='int8'), labels)))
    labels = tile_sequences(labels, tile_size=(1, 1))
    labels = np.squeeze(labels, axis=(2, 3)).astype(float)
    
    return images, labels

In [ ]:
def get_balanced_data(images, labels): 
    # Samples tiles (temporary), equal samples
    random.seed(1)
    index = np.concatenate([
        random.choice(np.where(labels == 0)[0], 2500),
        random.choice(np.where(labels == 3)[0], 2500)
    ])
    images_b = images[index]
    labels_b = labels[index]
    labels_b = np.where(labels_b == 3, 1.0, 0.0) # Converts to float
    del index
    
    return images_b, labels_b

In [ ]:
def get_annotation_statistics(CITY):
    temp = search_data(pattern(city=CITY, type='label'))
    dates = list(map(lambda x: x.split("_")[1], temp))
    years = list(set(list(map(lambda x: x.split("-")[0], dates))))
    years.sort()

    for year in years:
        temp2 = [label for label in temp if year in label]
        total_images = len(temp2)
        temp2 = np.array(list(map(lambda x: read_raster(x, dtype='int8'), temp2)))
        temp2 = tile_sequences(temp2, tile_size=(1, 1))
        temp2 = np.squeeze(temp2, axis=(2, 3)).astype(float)

        n, t, c = temp2.shape
        temp2=temp2.reshape(n*t, c)
        temp2 = np.where(temp2 == 3, 1.0, 0.0) # Converts to float

        non_damaged = np.unique(temp2, return_counts=True)[1][0]
        total = temp2.shape[0]
        damaged = total - non_damaged

        print(f"{year}: {total_images} image(s); {damaged} out of {total}, i.e, {np.round(damaged/total * 100, 2)}% patches damaged.")

        del damaged, total, non_damaged, n, t, c, temp2, total_images
    del temp, dates, years
    

## CNN for balanced data

In [ ]:
images, labels = get_images_and_labels(CITY)
images, labels = get_balanced_data(images, labels)

In [ ]:
get_annotation_statistics(CITY)

In [ ]:
# Reshape for convolutional neural net
n, t, h, w, d = images.shape
images = images.reshape(n*t, h, w,d)

n, t, c = labels.shape
labels = labels.reshape(n*t, c)
# labels = np.where(labels == 3, 1.0, 0.0) # Converts to float

del n, t, h, w, d, c

In [ ]:
np.unique(labels)

In [ ]:
# Splits samples (separate test sample)
np.random.seed(1)
samples_size = dict(train=0.8, valid=0.1, test=0.1)
index = np.random.choice(np.arange(len(samples_size)) + 1, len(images), p=list(samples_size.values()))
images_train, images_valid, images_test = sample_split(images, index)
labels_train, labels_valid, labels_test = sample_split(labels, index)
del index

In [ ]:
#%% AUGMENTATION
# Generator parameters
augmentation = dict(
#     rescale=1./255,
    horizontal_flip=True, 
#     vertical_flip=True,
    rotation_range=10, 
#     width_shift_range=0.1,
#     height_shift_range=0.1,
    brightness_range=[0.7,1.3],
    zoom_range=[0.7, 1.3],
    fill_mode='nearest'
)

# Initialises data generator
train_generator = preprocessing.image.ImageDataGenerator(**augmentation)
train_generator = train_generator.flow(images_train, labels_train, batch_size=32, shuffle=True, seed=1)

#%% MODELS

# Initialises model
args  = dict(shape=(128, 128, 3), filters=16, units=32, dropout=0.1) # ! Check parameters before run
model = models.convolutional_network(**args)
model.compile(optimizer='adam', loss='binary_focal_crossentropy', metrics='accuracy')


#%% OPTIMISATION
# Callbacks
train_callbacks = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=50, restore_best_weights=True),
    callbacks.BackupAndRestore(backup_dir='../models')
]



In [ ]:
# Estimates parameters
training = model.fit(
    train_generator,
    steps_per_epoch=images_train.shape[0] // 32,
    validation_data=(images_valid, labels_valid),
    epochs=100,
    verbose=1,
#     callbacks=train_callbacks
)

In [ ]:
# probs = model.predict_proba(images_test)
preds = model.predict(images_test)
precision, recall, _ = precision_recall_curve(labels_test, preds)

fig, ax = plt.subplots(1,1,dpi=100, figsize=(10,5))
plt.plot(recall, precision);
ax.set_title('Precision - Recall Curve for test sample');
ax.set_xlabel('Precision');
ax.set_ylabel('Recall');


In [ ]:
roc_auc_score(labels_test, preds)

### Optimize script: master

In [ ]:
#%% HEADER

# Modules
import numpy as np
import pandas as pd
import itertools
import destruction_models as models

from destruction_utilities import *
from numpy import random
from matplotlib import pyplot
from keras import callbacks, preprocessing
from os import path

#%% FUNCTIONS

def display_history(history:dict, stats:list=['accuracy', 'loss']) -> None:
    '''Displays model training history'''
    fig, axs = pyplot.subplots(nrows=1, ncols=2, figsize=(10, 5))
    for ax, stat in zip(axs.ravel(), stats):
        ax.plot(history[stat])
        ax.plot(history[f'val_{stat}'])
        ax.set_title(f'Training {stat}', fontsize=15)
        ax.set_ylabel('Accuracy')
        ax.set_xlabel('Epoch')
        ax.legend(['Training sample', 'Validation sample'], frameon=False)
    pyplot.tight_layout(pad=2.0)
    pyplot.show()

#%% DECLARATION
CITY = 'daraa'

#%% READS DATA
# Reads images (subset)
images = search_data(pattern(city=CITY, type='image'))
images = np.array(list(map(read_raster, images)))
images = tile_sequences(images, tile_size=(128, 128))

# Reads labels (subset)
labels = search_data(pattern(city=CITY, type='label'))
labels = np.array(list(map(lambda x: read_raster(x, dtype='int8'), labels)))
labels = tile_sequences(labels, tile_size=(1, 1))
labels = np.squeeze(labels, axis=(2, 3)).astype(float)

# # Samples tiles (temporary), equal samples
# random.seed(1)
# index = np.concatenate([
#     random.choice(np.where(labels == 0)[0], 5000),
#     random.choice(np.where(labels == 3)[0], 5000)
# ])
# images = images[index]
# labels = labels[index]
# labels = np.where(labels == 3, 1.0, 0.0) # Converts to float
# del index

# Reshape for convolutional neural net
n, t, h, w, d = images.shape
images = images.reshape(n*t, h, w,d)


n, t, c = labels.shape
labels=labels.reshape(n*t, c)
labels = np.where(labels == 3, 1.0, 0.0) # Converts to float


# Splits samples (separate test sample)
np.random.seed(1)
samples_size= dict(train=0.8, valid=0.1, test=0.1)
index   = np.random.choice(np.arange(len(samples_size)) + 1, len(images), p=list(samples_size.values()))
images_train, images_valid, images_test = sample_split(images, index)
labels_train, labels_valid, labels_test = sample_split(labels, index)
del index


#%% AUGMENTATION

# Generator parameters
augmentation = dict(
    rescale=1./255,
    horizontal_flip=True, 
    vertical_flip=True,
    rotation_range=10, 
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.9,1.1],
    zoom_range=[0.9, 1.1],
    fill_mode='nearest'
)

# Initialises data generator
train_generator = preprocessing.image.ImageDataGenerator(**augmentation)
train_generator = train_generator.flow(images_train, labels_train, batch_size=32, shuffle=True, seed=1)
# del images_train, labels_train

#%% MODELS

# Initialises model
args  = dict(shape=(128, 128, 3), filters=16, units=32, dropout=0.1) # ! Check parameters before run
model = models.convolutional_network(**args)
model.compile(optimizer='adam', loss='binary_focal_crossentropy', metrics='accuracy')
# display_structure(model, path.join(paths['models'], 'convolutional_network.html'))


#%% OPTIMISATION

# Callbacks
train_callbacks = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True),
    callbacks.BackupAndRestore(backup_dir='../models')
]

# Estimates parameters
training = model.fit(
    train_generator,
#     steps_per_epoch=samples_size['train'] // 32,
    validation_data=(images_valid, labels_valid),
    epochs=100,
    verbose=1,
    callbacks=train_callbacks
)


In [ ]:
labels.shape